# UK Online Retail Performance Analysis
Source dataset: [UCI Online Retail](https://archive.ics.uci.edu/dataset/352/online+retail) (Chen, D. 2015, CC BY 4.0)

This notebook builds the SQLite database from the raw CSV, runs the SQL cleaning + analysis scripts,
and regenerates every chart used in `README.md`.

In [1]:
import sqlite3, pandas as pd, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

navy='#1f3b57'; teal='#2a9d8f'; coral='#e76f51'; gold='#e9c46a'; grey='#6c757d'
plt.rcParams.update({'font.size': 11, 'figure.dpi': 150})

## 1. Build the database from raw data + SQL scripts

In [2]:
conn = sqlite3.connect('../online_retail.db')

df = pd.read_csv('../data/Online_Retail_Raw.csv')
df.to_sql('raw_transactions', conn, if_exists='replace', index=False)

conn.executescript(open('../sql/01_schema.sql').read())
conn.executescript(open('../sql/02_cleaning.sql').read())
conn.commit()

for t in ['geo_lookup','products','customers','orders','order_lines','excluded_lines']:
    n = conn.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
    print(t, n)

geo_lookup 39
products 3947
customers 4373
orders 25897
order_lines 536636
excluded_lines 5273


## 2. Sales trend — monthly revenue, orders, AOV

In [3]:
monthly = pd.read_sql('''
SELECT
    strftime('%Y-%m', o.invoice_date) AS year_month,
    ROUND(SUM(ol.line_revenue), 2)     AS revenue,
    COUNT(DISTINCT o.invoice_no)       AS orders,
    ROUND(SUM(ol.line_revenue) * 1.0 / COUNT(DISTINCT o.invoice_no), 2) AS aov
FROM orders o
JOIN order_lines ol ON ol.invoice_no = o.invoice_no
WHERE o.status_id = 1
GROUP BY year_month
ORDER BY year_month
''', conn)
monthly

,year_month,revenue,orders,aov
0,2010-12,778608.36,1551,502.00
1,2011-01,672192.41,1081,621.82
2,2011-02,509167.87,1093,465.84
3,2011-03,692015.94,1440,480.57
4,2011-04,516709.79,1235,418.39
5,2011-05,741711.74,1668,444.67
6,2011-06,739426.88,1525,484.87
7,2011-07,689947.74,1452,475.17
8,2011-08,726005.16,1340,541.79
9,2011-09,1031400.36,1819,567.02


In [4]:
fig, ax1 = plt.subplots(figsize=(10,5))
ax1.plot(monthly['year_month'], monthly['revenue']/1000, color=navy, marker='o', linewidth=2)
ax1.set_ylabel('Revenue (£000s)')
ax1.set_title('Monthly Revenue Trend (Dec 2010 - Dec 2011)')
ax1.tick_params(axis='x', rotation=45)
for i, label in enumerate(ax1.get_xticklabels()):
    if i % 2 != 0: label.set_visible(False)
plt.tight_layout(); plt.savefig('../visuals/01_monthly_revenue.png'); plt.show()

In [5]:
fig, ax1 = plt.subplots(figsize=(10,5))
ax1.plot(monthly['year_month'], monthly['orders'], color=teal, marker='o', linewidth=2)
ax1.set_ylabel('Number of Orders')
ax1.set_title('Monthly Order Count Trend')
ax1.tick_params(axis='x', rotation=45)
for i, label in enumerate(ax1.get_xticklabels()):
    if i % 2 != 0: label.set_visible(False)
plt.tight_layout(); plt.savefig('../visuals/02_monthly_orders.png'); plt.show()

In [6]:
fig, ax1 = plt.subplots(figsize=(10,5))
ax1.plot(monthly['year_month'], monthly['aov'], color=coral, marker='o', linewidth=2)
ax1.set_ylabel('Average Order Value (£)')
ax1.set_title('Average Order Value (AOV) Trend')
ax1.tick_params(axis='x', rotation=45)
for i, label in enumerate(ax1.get_xticklabels()):
    if i % 2 != 0: label.set_visible(False)
plt.tight_layout(); plt.savefig('../visuals/03_monthly_aov.png'); plt.show()

## 3. Product performance — top 10 by revenue

In [7]:
top10 = pd.read_sql('''
SELECT p.description, p.stock_code,
       ROUND(SUM(ol.line_revenue), 2) AS total_revenue,
       SUM(ol.quantity) AS total_units
FROM order_lines ol
JOIN orders o ON o.invoice_no = ol.invoice_no
JOIN products p ON p.stock_code = ol.stock_code
WHERE o.status_id = 1
GROUP BY p.stock_code
ORDER BY total_revenue DESC
LIMIT 10
''', conn)
top10

,description,stock_code,total_revenue,total_units
0,REGENCY CAKESTAND 3 TIER,22423,174484.74,13879
1,"PAPER CRAFT , LITTLE BIRDIE",23843,168469.60,80995
2,WHITE HANGING HEART T-LIGHT HOLDER,85123A,104518.80,37660
3,PARTY BUNTING,47566,99504.33,18295
4,JUMBO BAG RED RETROSPOT,85099B,94340.05,48474
5,MEDIUM CERAMIC TOP STORAGE JAR,23166,81700.92,78033
6,RABBIT NIGHT LIGHT,23084,66964.99,30788
7,PAPER CHAIN KIT 50'S CHRISTMAS,22086,64952.29,19355
8,ASSORTED COLOUR BIRD ORNAMENT,84879,59094.93,36461
9,CHILLI LIGHTS,79321,54117.76,10306


In [8]:
fig, ax = plt.subplots(figsize=(9,6))
labels = [d[:32] for d in top10['description'][::-1]]
ax.barh(labels, top10['total_revenue'][::-1]/1000, color=navy)
ax.set_xlabel('Total Revenue (£000s)')
ax.set_title('Top 10 Products by Revenue')
plt.tight_layout(); plt.savefig('../visuals/04_top10_products.png'); plt.show()

## 4. Cancellation rates by product

In [9]:
cancel = pd.read_sql('''
SELECT p.description, p.stock_code,
       SUM(CASE WHEN o.status_id = 2 THEN 1 ELSE 0 END) AS cancelled_lines,
       COUNT(*) AS total_lines,
       ROUND(100.0 * SUM(CASE WHEN o.status_id = 2 THEN 1 ELSE 0 END) / COUNT(*), 2) AS cancel_rate_pct
FROM order_lines ol
JOIN orders o ON o.invoice_no = ol.invoice_no
JOIN products p ON p.stock_code = ol.stock_code
GROUP BY p.stock_code
HAVING total_lines >= 30
ORDER BY cancel_rate_pct DESC
LIMIT 15
''', conn)

fig, ax = plt.subplots(figsize=(9,6))
labels = [d[:32] for d in cancel['description'][::-1]]
ax.barh(labels, cancel['cancel_rate_pct'][::-1], color=coral)
ax.set_xlabel('Cancellation Rate (%)')
ax.set_title('Top 15 Products by Cancellation Rate (min. 30 order lines)')
plt.tight_layout(); plt.savefig('../visuals/05_cancellation_rate.png'); plt.show()

## 5. Regional results

In [10]:
region = pd.read_sql('''
SELECT g.region,
       ROUND(SUM(ol.line_revenue), 2) AS revenue,
       COUNT(DISTINCT o.invoice_no) AS orders,
       ROUND(100.0 * SUM(ol.line_revenue) / (SELECT SUM(line_revenue) FROM order_lines ol2
            JOIN orders o2 ON o2.invoice_no = ol2.invoice_no WHERE o2.status_id = 1), 2) AS pct_of_total
FROM order_lines ol
JOIN orders o ON o.invoice_no = ol.invoice_no
JOIN geo_lookup g ON g.country = o.country
WHERE o.status_id = 1
GROUP BY g.region
ORDER BY revenue DESC
''', conn)
region

,region,revenue,orders,pct_of_total
0,UK,8749722.47,17901,85.12
1,Western Europe,793143.33,1109,7.72
2,Ireland,276404.30,284,2.69
3,Rest of World,220846.57,126,2.15
4,Southern Europe,118311.31,197,1.15
5,Nordics,110160.19,131,1.07
6,Eastern Europe,10581.70,28,0.10


In [11]:
fig, ax = plt.subplots(figsize=(8,5))
ax.bar(region['region'], region['revenue']/1000, color=[navy,teal,coral,gold,grey,'#8d99ae','#adb5bd'][:len(region)])
ax.set_ylabel('Revenue (£000s)')
ax.set_title('Revenue by Region')
plt.xticks(rotation=30, ha='right')
plt.tight_layout(); plt.savefig('../visuals/06_region_revenue.png'); plt.show()

## 6. Customer segmentation — repeat vs. one-time, guest vs. registered

In [12]:
seg = pd.read_sql('''
WITH cust_orders AS (
    SELECT o.customer_id, COUNT(DISTINCT o.invoice_no) AS n_orders,
           SUM(ol.line_revenue) AS total_spend
    FROM orders o
    JOIN order_lines ol ON ol.invoice_no = o.invoice_no
    WHERE o.status_id = 1 AND o.customer_id <> 0
    GROUP BY o.customer_id
)
SELECT
    CASE WHEN n_orders = 1 THEN 'One-time' ELSE 'Repeat' END AS segment,
    COUNT(*) AS customers,
    ROUND(AVG(total_spend), 2) AS avg_customer_revenue,
    ROUND(AVG(total_spend * 1.0 / n_orders), 2) AS avg_order_value,
    ROUND(SUM(total_spend), 2) AS segment_revenue
FROM cust_orders
GROUP BY segment
''', conn)
seg

,segment,customers,avg_customer_revenue,avg_order_value,segment_revenue
0,One-time,1505,418.34,418.34,629594.67
1,Repeat,2829,2876.69,416.60,8138157.98


In [13]:
fig, axes = plt.subplots(1,2, figsize=(11,5))
axes[0].bar(seg['segment'], seg['avg_order_value'], color=[teal,navy])
axes[0].set_title('Average Order Value by Customer Segment')
axes[0].set_ylabel('AOV (£)')
axes[1].bar(seg['segment'], seg['customers'], color=[teal,navy])
axes[1].set_title('Registered Customers by Segment')
axes[1].set_ylabel('Number of Customers')
plt.tight_layout(); plt.savefig('../visuals/08_customer_segmentation.png'); plt.show()

## 7. Quarterly heat map — top 5 products

In [14]:
heat = pd.read_sql('''
WITH top5 AS (
    SELECT stock_code FROM (
        SELECT ol.stock_code, SUM(ol.line_revenue) rev
        FROM order_lines ol JOIN orders o ON o.invoice_no = ol.invoice_no
        WHERE o.status_id = 1
        GROUP BY ol.stock_code ORDER BY rev DESC LIMIT 5
    )
)
SELECT p.description,
       strftime('%Y', o.invoice_date) || '-Q' ||
       ((CAST(strftime('%m', o.invoice_date) AS INTEGER) + 2) / 3) AS quarter,
       ROUND(SUM(ol.line_revenue), 2) AS revenue
FROM order_lines ol
JOIN orders o ON o.invoice_no = ol.invoice_no
JOIN products p ON p.stock_code = ol.stock_code
WHERE o.status_id = 1 AND ol.stock_code IN (SELECT stock_code FROM top5)
GROUP BY p.description, quarter
ORDER BY p.description, quarter
''', conn)
pivot = heat.pivot(index='description', columns='quarter', values='revenue').fillna(0)
pivot

quarter,2010-Q4,2011-Q1,2011-Q2,2011-Q3,2011-Q4
description,,,,,
JUMBO BAG RED RETROSPOT,4019.15,20783.27,19248.79,25934.76,24354.08
"PAPER CRAFT , LITTLE BIRDIE",0.00,0.00,0.00,0.00,168469.60
PARTY BUNTING,1207.74,15759.28,42487.29,30674.30,9375.72
REGENCY CAKESTAND 3 TIER,27869.96,42182.09,37204.31,34140.69,33087.69
WHITE HANGING HEART T-LIGHT HOLDER,10435.36,25961.22,25915.02,21082.14,21125.06


In [15]:
fig, ax = plt.subplots(figsize=(9,5))
im = ax.imshow(pivot.values, cmap='YlOrRd', aspect='auto')
ax.set_xticks(range(len(pivot.columns))); ax.set_xticklabels(pivot.columns, rotation=45, ha='right')
ax.set_yticks(range(len(pivot.index))); ax.set_yticklabels([i[:28] for i in pivot.index])
for i in range(pivot.shape[0]):
    for j in range(pivot.shape[1]):
        v = pivot.values[i,j]
        ax.text(j, i, f"{v/1000:.0f}k", ha='center', va='center', fontsize=8,
                color='white' if v > pivot.values.max()*0.5 else 'black')
ax.set_title('Quarterly Revenue Heat Map — Top 5 Products by Revenue')
fig.colorbar(im, ax=ax, label='Revenue (£)')
plt.tight_layout(); plt.savefig('../visuals/09_quarterly_heatmap.png'); plt.show()

In [16]:
conn.close()